## Export VIDEO ELAIS confidence-map metadata

Create `example_export_confidence.ecsv` for Butler ingestion of the VIDEO ELAIS confidence maps.

The table follows the same format used for the VIDEO CDFS confidence maps: one row per VIRCAM detector (1--16) for every `_st_conf.fit` file.

### 1. Imports

In [1]:
from pathlib import Path

from astropy.io import fits
from astropy.table import Table


### 2. Input confidence maps

The downloaded VIDEO ELAIS confidence maps are stored directly in the local `data` directory.

In [2]:
base_dir = Path(".")
data_dir = base_dir / "data"

output_file = (
    base_dir / "example_export_confidence.ecsv"
)

# Confidence maps are stored directly in data/.
conf_files = sorted(
    data_dir.glob("*_st_conf.fit")
)

print(
    f"Found {len(conf_files)} confidence maps"
)

if not conf_files:
    raise RuntimeError(
        f"No *_st_conf.fit files found in {data_dir}"
    )

conf_files[:5]

Found 1957 confidence maps


[PosixPath('data/v20100707_00726_st_conf.fit'),
 PosixPath('data/v20100707_00737_st_conf.fit'),
 PosixPath('data/v20100707_00749_st_conf.fit'),
 PosixPath('data/v20100709_00695_st_conf.fit'),
 PosixPath('data/v20100709_00706_st_conf.fit')]

### 3. Extract Butler metadata

For each confidence map:

- read the VIRCAM band from `ESO INS FILT1 NAME`;
- read the exposure ID from `ESO DET EXP NO`;
- create one row for each of the 16 VIRCAM detectors;
- use the same `band` and `physical_filter` convention as the CDFS confidence metadata.

In [3]:
rows = []

for i, path in enumerate(conf_files, start=1):

    with fits.open(path, memmap=True) as hdul:
        primary_header = hdul[0].header
        detector_header = hdul[1].header

        band = str(
            primary_header["ESO INS FILT1 NAME"]
        ).strip()

        exposure = int(
            detector_header["ESO DET EXP NO"]
        )

    if band == "K":
        physical_filter = "VIRCAM-Ks"
    else:
        physical_filter = f"VIRCAM-{band}"

    # The confidence files are stored directly in the
    # VIDEO ELAIS data directory, with no date subdirectories.
    filename = (
        "../../../dmu0/dmu0_VISTA/"
        f"dmu0_VIDEO_ELAIS/data/{path.name}"
    )

    for detector in range(1, 17):
        rows.append(
            (
                filename,
                "VIRCAM",
                band,
                physical_filter,
                exposure,
                detector,
            )
        )

    if i % 200 == 0 or i == len(conf_files):
        print(
            f"Processed {i}/{len(conf_files)} files"
        )

Processed 200/1957 files
Processed 400/1957 files
Processed 600/1957 files
Processed 800/1957 files
Processed 1000/1957 files
Processed 1200/1957 files
Processed 1400/1957 files
Processed 1600/1957 files
Processed 1800/1957 files
Processed 1957/1957 files


### 4. Build and inspect the ECSV table

In [4]:
confidence_table = Table(
    rows=rows,
    names=(
        "filename",
        "instrument",
        "band",
        "physical_filter",
        "exposure",
        "detector",
    ),
)

print(f"Confidence maps : {len(conf_files)}")
print(f"ECSV rows       : {len(confidence_table)}")
print(f"Expected rows   : {16 * len(conf_files)}")
print(f"Bands           : {sorted(set(confidence_table['band']))}")

confidence_table[:20]


Confidence maps : 1957
ECSV rows       : 31312
Expected rows   : 31312
Bands           : [np.str_('H'), np.str_('J'), np.str_('Ks'), np.str_('Y'), np.str_('Z')]


filename,instrument,band,physical_filter,exposure,detector
str74,str6,str2,str9,int64,int64
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,1
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,2
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,3
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,4
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,5
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,6
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,7
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,8
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,9


### 5. Validate detector rows

Each confidence map should contribute exactly 16 detector rows.

In [5]:
assert len(confidence_table) == 16 * len(conf_files)
assert set(confidence_table["instrument"]) == {"VIRCAM"}
assert set(confidence_table["detector"]) == set(range(1, 17))

print("Validation passed.")


Validation passed.


### 6. Write `example_export_confidence.ecsv`

In [6]:
confidence_table.write(
    output_file,
    format="ascii.ecsv",
    overwrite=True,
)

print(f"Wrote: {output_file}")


Wrote: example_export_confidence.ecsv


### 7. Read back the written file

This final check confirms that the saved ECSV can be read correctly before using it with Butler.

In [7]:
check = Table.read(output_file, format="ascii.ecsv")

print(f"Rows read back: {len(check)}")
check[:20]


Rows read back: 31312


filename,instrument,band,physical_filter,exposure,detector
str74,str6,str2,str9,int64,int64
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,1
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,2
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,3
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,4
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,5
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,6
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,7
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,8
../../../dmu0/dmu0_VISTA/dmu0_VIDEO_ELAIS/data/v20100707_00726_st_conf.fit,VIRCAM,Z,VIRCAM-Z,133741,9
